모델 배포 개론 08  
Last modified : 2026.03   
작성 : 박광성 (모두의연구소)  
수정 : 김지성 (모두의연구소)  

In [3]:
!pip install uvicorn fastapi transformers torch pillow requests streamlit


In [4]:
# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
# 노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 작업 디렉터리를 app/ 가 있는 위치로 맞춥니다 (notebooks/ 안에서 열어도 동작).
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# 코드를 저장할 폴더를 미리 만들어 둡니다.
for _d in ('app', 'models', 'data', 'frontend'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드에서 uvicorn 서버를 띄웁니다.

    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 이미 있으면 먼저 멈추고 새로 띄웁니다.
    """
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다 (다른 노트북의 서버일 가능성).')
        print('   그 노트북에서 stop_server(8000)을 실행하거나 커널을 종료한 뒤, 이 셀을 다시 실행하세요.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        # Windows는 SelectorEventLoop, 그 외는 기본 이벤트 루프를 사용합니다.
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    # 모델 로드 때문에 기동이 느릴 수 있다 — 최대 5분 대기 (첫 실행은 다운로드 포함)
    for i in range(600):
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        if not thread.is_alive():
            print('서버 스레드가 종료됐습니다. 위 로그를 확인하세요.')
            return server
        if i > 0 and i % 20 == 0:
            print(f'  ... 모델 로드 중 ({i//2}초 경과)')
        time.sleep(0.5)
    print('5분 내에 서버가 시작되지 않았습니다. 위 로그를 확인하세요.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')


서버 도우미 준비 완료 (serve_in_thread, stop_server)


# Day 8 — 자율 프로젝트: 나만의 모델 서빙 서비스 만들기

---

> **오늘의 목표**
>
> Day 1~7에서 배운 기술을 조합하여, 본인이 관심 있는 도메인의 모델을 서빙하는 서비스를 직접 만듭니다.  
> 따라하기가 아닌, **스스로 설계하고 구현하는** 첫 경험입니다.

---



## 1. 프로젝트 요구사항

---

### 1.1 조건

Day 5(주택 가격 예측)와 Day 6~7(이미지 분류 / 챗봇)에서 만든 서비스와 **동일한 구조**를 기본 베이스로 합니다. 

```
필수 구현 항목:

1. FastAPI 백엔드
   - 추론 엔드포인트 (POST /predict)
   - Pydantic으로 입력 검증
   - 비동기 추론 (run_in_executor)

2. API Key 인증
   - Day 6의 auth.py 재사용

3. Streamlit 프론트엔드
   - 사용자 입력 → API 호출 → 결과 표시

4. 에러 처리
   - 잘못된 입력, 모델 에러 시 적절한 HTTP 상태 코드 반환
```



### 1.2 제한 사항

```
하지 않는 것:

- 모델 학습 (사전학습 모델을 가져다 씁니다)
- Docker 패키징 (MLOps 과정에서 다룹니다)
- 데이터베이스 연동
```



### 1.3 평가 기준

```
✅ 서버가 정상적으로 실행되는가?
✅ Swagger UI에서 추론이 동작하는가?
✅ API Key 없이 요청하면 401이 반환되는가?
✅ 잘못된 입력에 대해 적절한 에러 메시지가 나오는가?
✅ Streamlit UI에서 입력 → 결과 확인이 가능한가?
```

---

## 2. 모델 선택 가이드

---

### 2.1 Hugging Face에서 모델 찾기

[Hugging Face Models](https://huggingface.co/models)에서 사전학습 모델을 선택합니다.
모델 학습은 하지 않고, `from_pretrained()`으로 바로 사용할 수 있는 모델을 고릅니다.

**모델 선택 시 확인할 것:**

```
1. 태스크가 명확한가? (text-classification, image-classification, summarization 등)
2. 한국어를 지원하는가? (필수는 아니지만, 데모가 직관적입니다)
3. 모델 크기가 적당한가? (CPU 환경이면 500MB 이하를 권장합니다)
4. pipeline()으로 바로 사용 가능한가?
```



### 2.2 도메인별 추천 예시

아래는 예시일 뿐입니다. **본인이 관심 있는 도메인을 자유롭게 선택하세요.**

| 도메인 | 태스크 | 추천 모델 (예시) |
|---|---|---|
| 감정 분석 | `text-classification` | `snunlp/KR-FinBert-SC` |
| 뉴스 분류 | `text-classification` | 원하는 분류 모델 |
| 텍스트 요약 | `summarization` | `eenzeenee/t5-base-korean` |
| 번역 | `translation` | `Helsinki-NLP` 시리즈 |
| 이미지 분류 | `image-classification` | `google/vit-base-patch16` |
| 객체 탐지 | `object-detection` | `facebook/detr-resnet-50` |
| 질의 응답 | `question-answering` | 원하는 QA 모델 |



### 2.3 모델 동작 확인

모델을 선택했으면, **서버 코드를 작성하기 전에** 노트북에서 먼저 동작을 확인합니다.

In [ ]:
# 예시: 텍스트 감정 분석
from transformers import pipeline

# 본인이 선택한 모델로 교체하세요
classifier = pipeline("text-classification", model="snunlp/KR-FinBert-SC")

result = classifier("오늘 주가가 크게 올랐습니다")
print(result)
# [{'label': 'positive', 'score': 0.95}]

예시 코드입니다. 샘플 이미지 파일이 있어야 실행됩니다.

```python
# 예시: 이미지 분류
from transformers import pipeline

classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

result = classifier("test_image.jpg")
print(result)
# [{'label': 'tabby cat', 'score': 0.82}, ...]
```

> **이 셀이 정상 동작하면**, 서버 코드 작성으로 넘어갑니다.
> 여기서 에러가 나면, 모델을 바꾸거나 입력 형식을 확인하세요.



---

## 3. 프로젝트 뼈대 코드

---



### 3.1 폴더 구조

In [8]:
import os

dirs = ["app", "models", "frontend"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("프로젝트 구조:")
print("""
my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas.py           ← 입력/출력 스키마 정의 (직접 작성)
│   ├── model_service.py     ← 모델 로드 + 추론 함수 (직접 작성)
│   └── main.py              ← FastAPI 서버 (직접 작성)
│
├── 📁 frontend/
│   └── app.py               ← Streamlit UI (직접 작성)
│
└── requirements.txt
""")

프로젝트 구조:

my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas.py           ← 입력/출력 스키마 정의 (직접 작성)
│   ├── model_service.py     ← 모델 로드 + 추론 함수 (직접 작성)
│   └── main.py              ← FastAPI 서버 (직접 작성)
│
├── 📁 frontend/
│   └── app.py               ← Streamlit UI (직접 작성)
│
└── requirements.txt



### 3.2 auth.py — 재사용

In [12]:
%%writefile app/schemas.py
"""
한국어 금융 뉴스 감정 분석 서비스의 입력/출력 스키마입니다.
"""

from pydantic import BaseModel, Field


class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=500,
        description="감정 분석할 한국어 금융 뉴스 문장"
    )


class PredictResponse(BaseModel):
    success: bool
    label: str
    confidence: float


Overwriting app/schemas.py


### 3.3 schemas.py — 직접 작성

In [13]:
%%writefile app/schemas.py
"""
입력/출력 스키마를 정의하세요.

참고: Day 2 섹션 4 (Pydantic 기초), Day 5 섹션 3 (HousingRequest)

TODO:
  - 본인의 모델 입력에 맞는 Request 스키마를 정의하세요.
  - 모델 출력에 맞는 Response 스키마를 정의하세요.
  - 필수 필드와 선택 필드를 구분하세요.
  - 적절한 검증 규칙(타입, 범위, 길이 등)을 추가하세요.

예시 (텍스트 분류의 경우):
  class PredictRequest(BaseModel):
      text: str = Field(..., min_length=1, max_length=5000)

  class PredictResponse(BaseModel):
      success: bool
      label: str
      confidence: float
"""
from pydantic import BaseModel, Field

# ── 여기에 작성하세요 ──────────────────────────

Overwriting app/schemas.py


### 3.4 model_service.py — 직접 작성

In [14]:
%%writefile app/model_service.py
"""
모델 로드와 추론 함수를 정의하세요.

참고: Day 1 섹션 5 (모델 로드), Day 5 섹션 2 (HousingPredictor)

TODO:
  1. load_model(): 모델을 로드하여 반환합니다.
     - transformers의 pipeline() 또는 from_pretrained()을 사용합니다.
     - 섹션 2.3에서 확인한 코드를 여기에 옮기면 됩니다.

  2. predict(model, input_data): 입력을 받아 추론 결과를 반환합니다.
     - 입력 전처리가 필요하면 여기서 합니다.
     - 결과를 dict로 반환합니다.

예시 (텍스트 분류의 경우):
  def load_model():
      return pipeline("text-classification", model="모델이름")

  def predict(model, text: str) -> dict:
      result = model(text)
      return {"label": result[0]["label"], "confidence": result[0]["score"]}
"""

%%writefile app/model_service.py
"""
모델 로드와 추론 함수를 정의합니다.
"""

from transformers import pipeline


def load_model():
    """
    Hugging Face 모델을 불러옵니다.
    """
    model = pipeline(
        "text-classification",
        model="snunlp/KR-FinBert-SC"
    )
    return model


def predict(model, text: str) -> dict:
    """
    입력 문장을 받아 감정 분석 결과를 반환합니다.
    """
    result = model(text)

    first_result = result[0]

    return {
        "label": first_result["label"],
        "confidence": float(first_result["score"])
    }

Overwriting app/model_service.py


### 3.5 main.py — 직접 작성

%%writefile app/main.py
"""
FastAPI 서버를 정의합니다.
"""

import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, Depends, HTTPException

from app.auth import verify_api_key
from app.schemas import PredictRequest, PredictResponse
from app.model_service import load_model, predict


app = FastAPI(
    title="한국어 금융 뉴스 감정 분석 API",
    description="한국어 금융 뉴스 문장을 입력하면 감정을 분석합니다.",
    version="1.0.0"
)

model = None
executor = ThreadPoolExecutor(max_workers=2)


@app.on_event("startup")
async def startup_event():
    global model
    model = load_model()


@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model_loaded": model is not None
    }


@app.post("/predict", response_model=PredictResponse)
async def predict_sentiment(
    request: PredictRequest,
    user: str = Depends(verify_api_key)
):
    if model is None:
        raise HTTPException(
            status_code=503,
            detail="모델이 아직 로드되지 않았습니다."
        )

    try:
        loop = asyncio.get_running_loop()

        result = await loop.run_in_executor(
            executor,
            predict,
            model,
            request.text
        )

        return PredictResponse(
            success=True,
            label=result["label"],
            confidence=result["confidence"]
        )

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"예측 중 오류가 발생했습니다: {str(e)}"
        )

In [42]:
!sed -n '1,80p' app/main.py

"""
FastAPI 서버를 정의합니다.
"""

import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, Depends, HTTPException

from app.auth import verify_api_key
from app.schemas import PredictRequest, PredictResponse
from app.model_service import load_model, predict


app = FastAPI(
    title="한국어 금융 뉴스 감정 분석 API",
    description="한국어 금융 뉴스 문장을 입력하면 감정을 분석합니다.",
    version="1.0.0"
)

model = None
executor = ThreadPoolExecutor(max_workers=2)


@app.on_event("startup")
async def startup_event():
    global model
    model = load_model()


@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model_loaded": model is not None
    }


@app.post("/predict", response_model=PredictResponse)
async def predict_sentiment(
    request: PredictRequest,
    user: str = Depends(verify_api_key)
):
    if model is None:
        raise HTTPException(
            status_code=503,
            detail="모델이 아직 로드되지 않았습니다."
        )

    try:
        loo

In [43]:
!uvicorn app.main:app --host 0.0.0.0 --port 8000 --log-level debug

Traceback (most recent call last):
  File "/opt/conda/bin/uvicorn", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1524, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1445, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1308, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 877, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/uvicorn/main.py", line 441, in main
    run(
  File "/opt/conda/lib/python3.12/site-packages/uvicorn/main.py", line 609, in run
    config.load_app()
  File "/opt/conda/lib/python3

In [44]:
!sed -n '1,120p' app/schemas.py

"""
입력/출력 스키마를 정의하세요.

참고: Day 2 섹션 4 (Pydantic 기초), Day 5 섹션 3 (HousingRequest)

TODO:
  - 본인의 모델 입력에 맞는 Request 스키마를 정의하세요.
  - 모델 출력에 맞는 Response 스키마를 정의하세요.
  - 필수 필드와 선택 필드를 구분하세요.
  - 적절한 검증 규칙(타입, 범위, 길이 등)을 추가하세요.

예시 (텍스트 분류의 경우):
  class PredictRequest(BaseModel):
      text: str = Field(..., min_length=1, max_length=5000)

  class PredictResponse(BaseModel):
      success: bool
      label: str
      confidence: float
"""
from pydantic import BaseModel, Field

# ── 여기에 작성하세요 ──────────────────────────


In [45]:
%%writefile app/schemas.py
"""
한국어 금융 뉴스 감정 분석 서비스의 입력/출력 스키마입니다.
"""

from pydantic import BaseModel, Field


class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=500,
        description="감정 분석할 한국어 금융 뉴스 문장"
    )


class PredictResponse(BaseModel):
    success: bool
    label: str
    confidence: float

Overwriting app/schemas.py


In [46]:
from app.schemas import PredictRequest, PredictResponse

print(PredictRequest)
print(PredictResponse)

<class 'app.schemas.PredictRequest'>
<class 'app.schemas.PredictResponse'>


In [47]:
!uvicorn app.main:app --host 0.0.0.0 --port 8000 --log-level debug

Traceback (most recent call last):
  File "/opt/conda/bin/uvicorn", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1524, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1445, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1308, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 877, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/uvicorn/main.py", line 441, in main
    run(
  File "/opt/conda/lib/python3.12/site-packages/uvicorn/main.py", line 609, in run
    config.load_app()
  File "/opt/conda/lib/python3

In [48]:
%%writefile app/model_service.py
"""
모델 로드와 추론 함수를 정의합니다.
"""

from transformers import pipeline


def load_model():
    """
    Hugging Face 모델을 불러옵니다.
    """
    model = pipeline(
        "text-classification",
        model="snunlp/KR-FinBert-SC"
    )
    return model


def predict(model, text: str) -> dict:
    """
    입력 문장을 받아 감정 분석 결과를 반환합니다.
    """
    result = model(text)

    first_result = result[0]

    return {
        "label": first_result["label"],
        "confidence": float(first_result["score"])
    }

Overwriting app/model_service.py


In [49]:
!sed -n '1,120p' app/model_service.py

"""
모델 로드와 추론 함수를 정의합니다.
"""

from transformers import pipeline


def load_model():
    """
    Hugging Face 모델을 불러옵니다.
    """
    model = pipeline(
        "text-classification",
        model="snunlp/KR-FinBert-SC"
    )
    return model


def predict(model, text: str) -> dict:
    """
    입력 문장을 받아 감정 분석 결과를 반환합니다.
    """
    result = model(text)

    first_result = result[0]

    return {
        "label": first_result["label"],
        "confidence": float(first_result["score"])
    }


In [50]:
!sed -n '1,120p' app/model_service.py

"""
모델 로드와 추론 함수를 정의합니다.
"""

from transformers import pipeline


def load_model():
    """
    Hugging Face 모델을 불러옵니다.
    """
    model = pipeline(
        "text-classification",
        model="snunlp/KR-FinBert-SC"
    )
    return model


def predict(model, text: str) -> dict:
    """
    입력 문장을 받아 감정 분석 결과를 반환합니다.
    """
    result = model(text)

    first_result = result[0]

    return {
        "label": first_result["label"],
        "confidence": float(first_result["score"])
    }


In [52]:
!sed -n '1,120p' app/model_service.py

"""
모델 로드와 추론 함수를 정의합니다.
"""

from transformers import pipeline


def load_model():
    """
    Hugging Face 모델을 불러옵니다.
    """
    model = pipeline(
        "text-classification",
        model="snunlp/KR-FinBert-SC"
    )
    return model


def predict(model, text: str) -> dict:
    """
    입력 문장을 받아 감정 분석 결과를 반환합니다.
    """
    result = model(text)

    first_result = result[0]

    return {
        "label": first_result["label"],
        "confidence": float(first_result["score"])
    }


In [53]:
from app.model_service import load_model, predict

print(load_model)
print(predict)

<function load_model at 0x78cab5a98ea0>
<function predict at 0x78cab5a98c20>


In [54]:
!grep -R "%%writefile" app frontend

In [57]:
!uvicorn app.main:app --host 0.0.0.0 --port 8000 --log-level debug

INFO:     Started server process [861]
INFO:     Waiting for application startup.
Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 13695.96it/s]
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
^C
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [861]


In [60]:
serve_in_thread("app.main:app", port=8000)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

서버 실행됨: http://127.0.0.1:8000


In [61]:
import requests

response = requests.get("http://localhost:8000/health", timeout=30)

print("상태 코드:", response.status_code)
print("응답 내용:", response.json())

상태 코드: 200
응답 내용: {'status': 'ok', 'model_loaded': True}


In [62]:
import requests

response = requests.post(
    "http://localhost:8000/predict",
    json={"text": "오늘 주가가 크게 올랐습니다."},
    headers={"X-API-Key": "test-key-001"},
    timeout=60
)

print("상태 코드:", response.status_code)
print("응답 내용:", response.json())

상태 코드: 200
응답 내용: {'success': True, 'label': 'positive', 'confidence': 0.99845290184021}


# 3.6 frontend/app.py — 직접 작성 

In [63]:
%%writefile frontend/app.py
"""
Streamlit 프론트엔드입니다.

사용자가 한국어 금융 뉴스 문장을 입력하면
FastAPI 서버로 문장을 보내고,
AI 감정 분석 결과를 화면에 보여줍니다.
"""

import streamlit as st
import requests


API_URL = "http://localhost:8000/predict"


st.title("한국어 금융 뉴스 감정 분석 서비스")

st.write(
    "한국어 금융 뉴스 문장을 입력하면 AI가 긍정, 부정, 중립 감정을 분석합니다."
)


api_key = st.sidebar.text_input(
    "API Key",
    value="test-key-001",
    type="password"
)


text = st.text_area(
    "분석할 문장을 입력하세요",
    value="오늘 주가가 크게 올랐습니다.",
    height=120
)


if st.button("감정 분석하기"):

    if not text.strip():
        st.warning("문장을 입력해 주세요.")

    else:
        headers = {
            "X-API-Key": api_key
        }

        payload = {
            "text": text
        }

        try:
            response = requests.post(
                API_URL,
                json=payload,
                headers=headers,
                timeout=60
            )

            if response.status_code == 200:
                result = response.json()

                st.success("분석 성공!")
                st.write("### 분석 결과")
                st.write(f"감정 라벨: **{result['label']}**")
                st.write(f"확신 점수: **{result['confidence']:.4f}**")

            else:
                st.error(f"오류 발생: {response.status_code}")

                try:
                    st.write(response.json())
                except Exception:
                    st.write(response.text)

        except Exception as e:
            st.error("서버 연결 오류가 발생했습니다.")
            st.write(str(e))

Overwriting frontend/app.py


---

## 4. 작업 시간

---

### 4.1 권장 순서



```
Step 1. 모델 선택 + 노트북에서 동작 확인 (섹션 2.3)
        → "이 모델이 내 입력에 대해 결과를 반환하는가?"

Step 2. schemas.py 작성
        → "입력과 출력의 형태를 정의"

Step 3. model_service.py 작성
        → "모델 로드 + 추론 함수"

Step 4. main.py 작성
        → "FastAPI 서버 조립"

Step 5. 서버 실행 + Swagger UI 테스트
        → "API가 동작하는가?"

Step 6. frontend/app.py 작성
        → "Streamlit UI 연결"
```



### 4.2 서버 실행 (Step 5에서 사용)

> ⚠️ **코드를 수정했는데 반영이 안 될 때 — 커널을 재시작하세요.**
>
> `app/main.py`, `app/model_service.py` 등을 고친 뒤 아래 셀을 다시 실행해도,
> 이미 메모리에 올라간 이전 코드가 남아 변경이 반영되지 않을 수 있습니다.
> 코드를 수정했다면 **커널 재시작(Kernel → Restart) 후 맨 위 셀부터 다시 실행**하세요.

In [65]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main:app", port=8000)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

서버 실행됨: http://127.0.0.1:8000


### 4.3 API 테스트 템플릿 (Step 5에서 사용)

In [26]:
%%writefile frontend/app.py
"""
Streamlit 프론트엔드입니다.

사용자가 한국어 금융 뉴스 문장을 입력하면
FastAPI 서버로 문장을 보내고,
AI 감정 분석 결과를 화면에 보여줍니다.

실행 방법:
  streamlit run frontend/app.py
"""

import streamlit as st
import requests


# FastAPI 서버 주소입니다.
API_URL = "http://localhost:8000/predict"


# 화면 제목
st.title("한국어 금융 뉴스 감정 분석 서비스")


# 서비스 설명
st.write(
    "한국어 금융 뉴스 문장을 입력하면 AI가 긍정, 부정, 중립 감정을 분석합니다."
)


# 사이드바에 API Key 입력창 만들기
api_key = st.sidebar.text_input(
    "API Key",
    value="test-key-001",
    type="password"
)


# 사용자가 문장을 입력하는 큰 입력창
text = st.text_area(
    "분석할 문장을 입력하세요",
    value="오늘 주가가 크게 올랐습니다.",
    height=120
)


# 버튼 만들기
if st.button("감정 분석하기"):

    # 문장이 비어 있는지 확인
    if not text.strip():
        st.warning("문장을 입력해 주세요.")

    else:
        # FastAPI 서버에 보낼 API Key
        headers = {
            "X-API-Key": api_key
        }

        # FastAPI 서버에 보낼 문장 데이터
        payload = {
            "text": text
        }

        try:
            # FastAPI 서버에 POST 요청 보내기
            response = requests.post(
                API_URL,
                json=payload,
                headers=headers,
                timeout=30
            )

            # 성공했을 때
            if response.status_code == 200:
                result = response.json()

                st.success("분석 성공!")

                st.write("### 분석 결과")
                st.write(f"감정 라벨: **{result['label']}**")
                st.write(f"확신 점수: **{result['confidence']:.4f}**")

            # 실패했을 때
            else:
                st.error(f"오류 발생: {response.status_code}")
                st.write(response.json())

        except Exception as e:
            st.error(f"서버 연결 오류: {e}")

Overwriting frontend/app.py


In [41]:
!sed -n '1,80p' app/main.py

"""
FastAPI 서버를 정의합니다.
"""

import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, Depends, HTTPException

from app.auth import verify_api_key
from app.schemas import PredictRequest, PredictResponse
from app.model_service import load_model, predict


app = FastAPI(
    title="한국어 금융 뉴스 감정 분석 API",
    description="한국어 금융 뉴스 문장을 입력하면 감정을 분석합니다.",
    version="1.0.0"
)

model = None
executor = ThreadPoolExecutor(max_workers=2)


@app.on_event("startup")
async def startup_event():
    global model
    model = load_model()


@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model_loaded": model is not None
    }


@app.post("/predict", response_model=PredictResponse)
async def predict_sentiment(
    request: PredictRequest,
    user: str = Depends(verify_api_key)
):
    if model is None:
        raise HTTPException(
            status_code=503,
            detail="모델이 아직 로드되지 않았습니다."
        )

    try:
        loo

In [66]:
serve_in_thread("app.main:app", port=8000)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

서버 실행됨: http://127.0.0.1:8000


In [34]:
!uvicorn app.main:app --host 0.0.0.0 --port 8000 --log-level debug

Traceback (most recent call last):
  File "/opt/conda/bin/uvicorn", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1524, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1445, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 1308, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/click/core.py", line 877, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/uvicorn/main.py", line 441, in main
    run(
  File "/opt/conda/lib/python3.12/site-packages/uvicorn/main.py", line 609, in run
    config.load_app()
  File "/opt/conda/lib/python3

In [67]:
import requests

response = requests.get("http://localhost:8000/health", timeout=10)

print("상태 코드:", response.status_code)
print("응답 내용:", response.json())

상태 코드: 200
응답 내용: {'status': 'ok', 'model_loaded': True}


In [ ]:
!streamlit run frontend/app.py



2026-06-18 09:15:54.310 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.176.8.6:8501
  External URL: http://35.201.173.44:8501



In [ ]:
serve_in_thread("app.main:app", port=8000)

In [71]:
import requests

API_URL = "http://localhost:8000"
HEADERS = {
    "X-API-Key": "test-key-001"
}

response = requests.post(
    f"{API_URL}/predict",
    json={
        "text": "오늘 주가가 크게 올랐습니다."
    },
    headers=HEADERS
)

print("상태 코드:", response.status_code)
print("응답 내용:", response.json())

상태 코드: 200
응답 내용: {'success': True, 'label': 'positive', 'confidence': 0.99845290184021}


In [70]:
import requests

response = requests.get("http://localhost:8000/health")

print(response.status_code)
print(response.json())

200
{'status': 'ok', 'model_loaded': True}


In [68]:
import requests

API_URL = "http://localhost:8000"

headers = {
    "X-API-Key": "test-key-001"
}

payload = {
    "text": "오늘 주가가 크게 올랐습니다."
}

response = requests.post(
    f"{API_URL}/predict",
    json=payload,
    headers=headers,
    timeout=60
)

print("상태 코드:", response.status_code)
print("응답 내용:", response.json())

상태 코드: 200
응답 내용: {'success': True, 'label': 'positive', 'confidence': 0.99845290184021}


In [ ]:
!streamlit run frontend/app.py --server.port 8501 --server.address 0.0.0.0

### 4.4 막혔을 때 참고할 Day



```
"스키마를 어떻게 정의하지?"        → Day 2 섹션 4, Day 5 섹션 3
"FastAPI 서버 구조가 기억 안 나"   → Day 5 섹션 3, Day 6 섹션 6
"run_in_executor 사용법?"         → Day 3 섹션 4
"인증 적용 방법?"                  → Day 6 섹션 2
"Streamlit에서 API 호출?"         → Day 4 섹션 6, Day 5 섹션 4
"에러 처리?"                      → Day 3 섹션 5
```

---

## 5. 발표 및 회고

---

### 5.1 발표 (개인당 5분)

```
발표 항목:
  1. 어떤 도메인/태스크를 선택했는가?
  2. 어떤 모델을 사용했는가? (선택 이유)
  3. 데모 시연 (Swagger UI 또는 Streamlit)
  4. 구현하면서 가장 어려웠던 부분은?
```

### 5.2 회고

```
스스로 돌아보기:
  - Day 1~7 교안 없이 코드를 작성할 수 있었는가?
  - 어떤 부분에서 교안을 다시 찾아봤는가?
  - 다음에 다시 만든다면 무엇을 다르게 하겠는가?
```

---

## 6. 8일간의 여정 정리

---



### 6.1 Day 1의 문제 → Day 8의 해결

```
Day 1의 문제                           해결한 Day
──────────────────────────────        ──────────
라이브러리가 없음                       Day 1: requirements.txt
모델 구조 코드 필요                     Day 1: model_utils.py 모듈 분리
전처리 로직 누락                       Day 5/7: 전처리 파라미터 저장
비개발자가 사용할 수 없음               Day 4/5/7: Streamlit UI
누구나 API 호출 가능                   Day 6: API Key 인증
스스로 서비스를 만들 수 있는가?          Day 8: 자율 프로젝트 ✅
```



### 6.2 8일간 배운 기술 전체 지도

```
Day 1: 환경 세팅 + 모델 직렬화          "모델을 저장하고 불러온다"
Day 2: FastAPI + Pydantic              "모델을 API로 감싼다"
Day 3: 비동기 처리 + 에러/로깅          "안정적으로 돌아가게 한다"
Day 4: Streamlit + 시스템 아키텍처      "누구나 쓸 수 있게 한다"
Day 5: [프로젝트 1] 정형 데이터 서비스   "따라하며 조립한다"
Day 6: 인증 + 파일 업로드               "보안과 비정형 데이터를 다룬다"
Day 7: [프로젝트 2] 텍스트/이미지 서비스  "패턴을 반복하며 익힌다"
Day 8: [자율 프로젝트] 나만의 서비스      "스스로 만든다"
```



### 6.3 Next Step: MLOps로 가는 길

```
이 과정에서 배운 것:                 다음 과정에서 배울 것:
──────────────────                  ──────────────────
수동으로 서버 실행                    → Docker로 패키징
단일 서버에서 실행                    → 클라우드 배포 (AWS, GCP)
코드 변경 시 수동 재시작              → CI/CD 파이프라인 (자동 빌드/배포)
모델 버전 1개                        → 모델 버전 관리 (MLflow, DVC)
수동 모니터링 (로그 확인)             → 자동 모니터링 (Prometheus, Grafana)
```



> **"코드를 고칠 때마다 매번 서버를 재시작해야 하나요?"**
>
> 그 질문의 답이 MLOps입니다.
> CI/CD 파이프라인이 코드 변경을 감지하면 자동으로 빌드, 테스트, 배포합니다.
> 여러분은 코드를 커밋하기만 하면 됩니다.

---


## Day 1~7은 부품을 배우는 시간이고,
## Day 8은 그 부품을 조립해서 나만의 AI 서비스를 만드는 시간이였습니다. 

모델 선택
↓
모델 테스트
↓
schemas.py 작성
↓
model_service.py 작성
↓
main.py 작성
↓
FastAPI 서버 실행
↓
API Key 인증 적용
↓
Streamlit 화면 연결 

스스로 서비스를 만들 수 있는가? 순서대로 진행해서 그 동안 수업을 빠졌는데도,
자세한 설명대로 해보니 스스로 서비스를 만들 수 있게 되어서 무지 기쁩니다. (  AI 가 많이 도와 주었지요 ) 

| 배운 내용                    | 이번 과제에서 한 일                |
| ------------------------ | -------------------------- |
| Day 1 환경 세팅              | 필요한 라이브러리와 모델 준비           |
| Day 2 FastAPI + Pydantic | `schemas.py`, `main.py` 작성 |
| Day 3 비동기 처리             | `run_in_executor` 사용       |
| Day 4 Streamlit          | `frontend/app.py` 작성       |
| Day 5 프로젝트 구조            | 파일별 역할 분리                  |
| Day 6 인증                 | `X-API-Key` 적용             |
| Day 7 텍스트 서비스 패턴         | 한국어 문장 감정 분석               |
| Day 8 자율 프로젝트            | 금융 뉴스 감정 분석 서비스 완성         |


8일간의 과정은 AI 모델 하나를 단순히 실행하는 것에서 시작해서,
그 모델을 실제 사용자가 쓸 수 있는 서비스로 만드는 흐름이었습니다.

Day 1에서는 모델과 라이브러리 환경을 준비하는 법
Day 2에서는 FastAPI와 Pydantic을 통해 모델을 API로 감싸는 법
Day 3에서는 서버가 안정적으로 동작하도록 비동기 처리와 에러 처리
Day 4에서는 Streamlit을 사용해 비개발자도 사용할 수 있는 화면
Day 6에서는 API Key 인증을 적용해 아무나 API를 호출하지 못하게 하는 방법

이번 Day 8 프로젝트에서는 이 모든 내용을 종합해서
한국어 금융 뉴스 감정 분석 서비스를 구현했습니다.
사용자는 Streamlit 화면에 문장을 입력하고,
FastAPI 서버는 API Key를 확인한 뒤 모델을 실행해서 감정 분석 결과를 반환합니다.

이번 과제를 통해 AI 모델을 단순히 실행하는 것과
AI 서비스를 만드는 것은 다르다는 것을 배웠습니다.
AI 서비스는 모델뿐 아니라 입력 검증, 서버 구조, 인증, 에러 처리, 사용자 화면까지 함께 필요하다는 것을 알게 되었습니다.

내가 직접 서버 실행
내가 직접 테스트
내가 직접 수정
내가 직접 재시작

코드 수정
↓
GitHub에 올림
↓
자동 테스트
↓
자동 빌드
↓
자동 배포
↓
자동 모니터링

MLOps = AI 서비스를 공장처럼 안정적으로 운영하는 방법

MLOps는 모델을 한 번 실행하는 기술이 아니라,
모델을 계속 안정적으로 운영하고 개선하는 체계입니다.


# ✅ Day 8 최종 체크포인트

```
# Q1. 본인의 프로젝트에서 Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?

class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=500
    )
1. text 값이 없는 요청
2. 빈 문자열 입력
3. 500자를 넘는 너무 긴 문장
4. 문자열이 아닌 잘못된 타입의 입력 

예를 들어 사용자가 아래처럼 보내면 문제가 됩니다.
{
  "text": ""
}
또는 :
{
  "wrong_name": "오늘 주가가 올랐습니다."
}
이런 입력은 API가 정상적으로 처리하지 않고, FastAPI/Pydantic이 먼저 잘못된 요청으로 막아줍니다.


# Q2. Depends(verify_api_key)를 제거하면 어떤 위험이 있습니까?

Depends(verify_api_key)를 제거하면 API Key 인증이 사라집니다.

현재 코드는 이런 구조입니다.
user: str = Depends(verify_api_key)
API를 사용하려는 사람이 올바른 출입증을 가지고 있는지 확인하는 부분

이 부분을 제거하면 다음과 같은 위험이 있습니다.
1. 아무나 API를 호출할 수 있습니다.
2. 모델 서버가 불필요하게 많이 사용될 수 있습니다.
3. 서버 비용이 증가할 수 있습니다.
4. 악의적인 사용자가 반복 요청을 보낼 수 있습니다.
5. 서비스 안정성이 떨어질 수 있습니다.

# 즉, API Key 인증은 서비스 입구의 잠금장치입니다.
이 잠금장치를 제거하면 누구나 들어와서 서비스를 사용할 수 있습니다.


# Q3. run_in_executor를 사용한 이유는 무엇입니까?

run_in_executor를 사용한 이유는 AI 모델 추론이 시간이 걸릴 수 있기 때문입니다.

AI 모델은 문장을 분석할 때 계산을 많이 합니다.
이 작업이 오래 걸리면 FastAPI 서버가 잠시 멈춘 것처럼 느껴질 수 있습니다.

그래서 코드에서 이렇게 작성했습니다.

result = await loop.run_in_executor(
    executor,
    predict,
    model,
    request.text
)

무거운 AI 분석 작업을 서버의 메인 흐름에서 직접 하지 않고,
옆 작업실에 맡겨서 실행하는 방식

비유하면:

FastAPI 서버 = 접수 데스크
AI 모델 추론 = 시간이 걸리는 전문 분석 작업
run_in_executor = 접수 데스크가 멈추지 않도록 분석실에 일을 넘기는 방법

그래서 run_in_executor를 사용하면 서버가 더 안정적으로 요청을 처리할 수 있습니다.

Q4. Day 1~8 중 가장 많이 참고한 Day는 어디였습니까? 왜?
가장 많이 참고한 Day는 Day 5와 Day 6입니다.

Day 5는 FastAPI 프로젝트 구조와 API 서버 조립 방식에 도움이 되었습니다.

schemas.py
model_service.py
main.py

처럼 파일 역할을 나누는 구조를 이해하는 데 필요했습니다.

Day 6은 API Key 인증을 적용하는 데 가장 많이 참고했습니다.

특히 아래 코드가 중요했습니다.
user: str = Depends(verify_api_key)

이 코드를 통해 /predict API를 아무나 호출하지 못하게 만들 수 있었습니다.

또한 Streamlit 화면을 만들 때는 Day 4와 Day 5도 참고했습니다.
Streamlit에서 사용자가 입력한 값을 받아 requests.post()로 FastAPI 서버에 보내는 구조를 이해하는 데 도움이 되었습니다.

정리하면:

Day 5: FastAPI 프로젝트 구조
Day 6: API Key 인증
Day 4/5: Streamlit UI와 API 호출
Day 3: run_in_executor와 에러 처리

를 주로 참고했습니다.

# Q5. 이 서비스를 실제로 배포하려면 추가로 무엇이 필요합니까?

이 서비스를 실제 사용자에게 배포하려면 지금보다 더 많은 준비가 필요합니다.

필요한 것은 다음과 같습니다.

1. Docker
2. 클라우드 서버
3. 환경변수 관리
4. 모델 파일 관리
5. 로그 관리
6. 모니터링
7. 보안 강화
8. CI/CD 자동 배포

1. Docker

현재는 내 노트북 환경에서만 실행됩니다.
Docker를 사용하면 다른 컴퓨터에서도 같은 환경으로 실행할 수 있습니다.

Docker = 서비스 실행 환경을 담는 포장 상자
2. 클라우드 서버

실제 사용자가 접속하려면 AWS, GCP, Azure 같은 클라우드 서버에 올려야 합니다.

내 컴퓨터에서 실행 = 나만 사용 가능
클라우드 서버에 배포 = 다른 사람도 접속 가능
3. 환경변수 관리

API Key 같은 중요한 값은 코드에 직접 쓰면 안 됩니다.
실제 배포에서는 .env나 클라우드 Secret Manager를 사용해야 합니다.

4. 로그와 모니터링

서비스가 잘 돌아가는지 확인해야 합니다.

몇 명이 사용했는지
에러가 얼마나 났는지
응답 시간이 얼마나 걸렸는지
모델이 너무 느려지지는 않았는지

를 확인해야 합니다.

5. CI/CD

코드를 고칠 때마다 수동으로 서버를 재시작하지 않고,
GitHub에 코드를 올리면 자동으로 테스트하고 배포되게 할 수 있습니다.

CI/CD = 코드 변경 → 자동 테스트 → 자동 배포

Q1. Pydantic 검증은 text 필드가 없거나, 빈 문자열이거나, 너무 긴 문장인 경우를 막아준다.

Q2. Depends(verify_api_key)를 제거하면 API Key 인증이 사라져 아무나 API를 호출할 수 있고, 서버 비용 증가와 악의적 요청 위험이 생긴다.

Q3. run_in_executor는 AI 모델 추론처럼 시간이 오래 걸리는 작업을 별도 실행 흐름에서 처리해 FastAPI 서버가 멈추지 않도록 하기 위해 사용했다.

Q4. Day 5와 Day 6을 가장 많이 참고했다. Day 5는 FastAPI 프로젝트 구조를 이해하는 데 도움이 되었고, Day 6은 API Key 인증을 적용하는 데 필요했다. Streamlit 연결은 Day 4와 Day 5도 참고했다.

Q5. 실제 배포를 위해서는 Docker, 클라우드 서버, 환경변수 관리, 로그/모니터링, 보안 강화, CI/CD 자동 배포, 모델 버전 관리가 추가로 필요하다.

이번 프로젝트를 통해 모델을 실행하는 것과 서비스를 만드는 것은 다르다는 것을 배웠다.
모델만 있으면 끝나는 것이 아니라,
입력 스키마, API 서버, 인증, 에러 처리, 프론트엔드 화면이 함께 있어야 실제 사용자가 쓸 수 있는 서비스가 된다.

아직 Docker, 클라우드 배포, CI/CD, 모델 버전 관리까지는 구현하지 못했지만,
이번 프로젝트는 MLOps로 가기 전 단계의 기초 서비스 구조를 직접 만들어 본 경험이었다.
다음에는 Docker로 실행 환경을 고정하고,
GitHub Actions 같은 CI/CD를 사용해 코드 변경 시 자동으로 테스트와 배포가 되도록 발전시키고 싶다.

---

### 📌 Day 8 요약 & 전체 과정 완료

```
오늘 한 일:
  ✅ Day 1~7의 기술을 조합하여 나만의 서비스를 직접 만들었습니다.
  ✅ 교안 없이 설계 → 구현 → 테스트를 경험했습니다.
  ✅ 8일간의 여정을 회고하고, MLOps로 가는 길을 확인했습니다.

8일간의 전체 성과:
  🎉 PyTorch / HuggingFace 모델을 API로 서빙할 수 있습니다.
  🎉 비개발자도 사용 가능한 웹 UI를 붙일 수 있습니다.
  🎉 인증, 에러 처리, 로깅으로 안정적인 서비스를 만들 수 있습니다.
  🎉 스스로 설계하고 구현할 수 있다는 자신감을 얻었습니다.
```

## 본 프로젝트는 Day 1~7에서 배운 모델 로드, FastAPI API화, Pydantic 스키마, 비동기 추론, API Key 인증, Streamlit UI 구성을 종합한 Day 8 자율 프로젝트이다.
한국어 금융 뉴스 감정 분석 모델을 FastAPI 서버로 제공하고, Streamlit 화면에서 사용자가 쉽게 문장을 입력해 결과를 확인할 수 있도록 구현했다.
이번 프로젝트를 통해 AI 모델을 실제 서비스 형태로 제공하기 위해서는 모델 자체뿐 아니라 서버 구조, 입력 검증, 인증, 에러 처리, 사용자 인터페이스가 모두 필요하다는 것을 배웠다.

### 제출
[제출 링크](https://forms.gle/AuzFv19QmyWh6rgc6)

1. FastAPI 서버 실행 성공
2. /health 테스트 성공
3. /predict 테스트 성공
4. Streamlit 실행 로그
5. README에 "환경상 Streamlit 브라우저 접속은 제한됨"이라고 회고 작성

#과제의 핵심은 모델 서빙 API가 작동하는가입니다.
Streamlit은 프론트엔드 연결이고, 현재는 코드가 아니라 접속 환경 문제로 링크가 오픈되지는 않았어요.
    

다음 내역을 MD 파일로 기록, 깃헙에 업로드하여 링크로 제출하시기 바랍니다  

1. 프로젝트 실행 내역 캡쳐와 설명
2. 각 섹션 체크포인트의 답변
3. 프로젝트 회고

수고하셨습니다!

# Day 8 자율 프로젝트 제출 정리

## 프로젝트명

한국어 금융 뉴스 감정 분석 서비스

## 프로젝트 개요

본 프로젝트는 한국어 금융 뉴스 문장을 입력하면 AI 모델이 문장의 감정을 분석해 주는 서비스이다.
사용자는 문장을 입력하고, 서버는 해당 문장을 분석하여 감정 라벨과 confidence 점수를 반환한다.

서비스는 FastAPI 백엔드와 Streamlit 프론트엔드로 구성하였다.
FastAPI는 모델 추론 API를 제공하고, Streamlit은 사용자가 쉽게 문장을 입력할 수 있는 화면 역할을 한다.

---

# 1. 프로젝트 실행 내역 캡처와 설명

## 1.1 모델 선택 및 노트북 동작 확인

### 실행 내용

Hugging Face의 `snunlp/KR-FinBert-SC` 모델을 사용하여 한국어 금융 문장의 감정 분석이 가능한지 확인하였다.

### 사용 코드 예시

```python
from transformers import pipeline

classifier = pipeline("text-classification", model="snunlp/KR-FinBert-SC")

result = classifier("오늘 주가가 크게 올랐습니다.")
print(result)
```

### 캡처 첨부

![모델 테스트 실행 결과](model_test.png)

### 설명

모델이 입력 문장에 대해 감정 분석 결과를 반환하는지 확인하였다.
이 단계는 선택한 모델이 프로젝트 목적에 맞게 동작하는지 검증하는 과정이다.

---

## 1.2 `schemas.py` 작성

### 실행 내용

FastAPI에서 사용할 입력과 출력의 형태를 Pydantic으로 정의하였다.

### 주요 코드

```python
class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=500,
        description="감정 분석할 한국어 금융 뉴스 문장"
    )


class PredictResponse(BaseModel):
    success: bool
    label: str
    confidence: float
```

### 캡처 첨부

![schemas.py 작성 결과](schemas_result.png)

### 설명

`PredictRequest`는 사용자가 서버에 보내는 입력 데이터의 형태를 정의한다.
`PredictResponse`는 서버가 사용자에게 반환하는 결과 데이터의 형태를 정의한다.

---

## 1.3 `model_service.py` 작성

### 실행 내용

모델을 불러오는 함수와 실제 추론을 실행하는 함수를 분리하여 작성하였다.

### 주요 코드

```python
def load_model():
    model = pipeline(
        "text-classification",
        model="snunlp/KR-FinBert-SC"
    )
    return model


def predict(model, text: str) -> dict:
    result = model(text)
    first_result = result[0]

    return {
        "label": first_result["label"],
        "confidence": float(first_result["score"])
    }
```

### 캡처 첨부

![model\_service.py 작성 결과](model_service_result.png)

### 설명

`load_model()`은 Hugging Face 모델을 불러오는 역할을 한다.
`predict()`는 입력 문장을 모델에 넣고 감정 라벨과 confidence 점수를 반환하는 역할을 한다.

---

## 1.4 `main.py` 작성

### 실행 내용

FastAPI 서버를 구성하고 `/health`, `/predict` 엔드포인트를 구현하였다.

### 주요 기능

* 서버 상태 확인 API: `GET /health`
* 감정 분석 API: `POST /predict`
* API Key 인증 적용
* `run_in_executor`를 이용한 추론 처리
* 에러 처리 적용

### 캡처 첨부

![main.py 작성 결과](main_result.png)

### 설명

`main.py`는 전체 FastAPI 서버의 중심 파일이다.
사용자의 요청을 받고, API Key를 검증한 뒤, 모델 추론 함수를 호출하여 결과를 반환한다.

---

## 1.5 FastAPI 서버 실행

### 실행 내용

FastAPI 서버를 실행하여 API가 동작하는지 확인하였다.

### 실행 코드

```python
serve_in_thread("app.main:app", port=8000)
```

또는 직접 실행 시:

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000
```

### 캡처 첨부

![FastAPI 서버 실행 화면](fastapi_server.png)

### 설명

서버 실행 로그에서 `Application startup complete`와 `Uvicorn running on http://0.0.0.0:8000` 메시지를 확인하였다.
이는 FastAPI 서버가 정상적으로 실행되었음을 의미한다.

---

## 1.6 `/health` API 테스트

### 실행 내용

서버가 정상적으로 켜져 있고 모델이 로드되었는지 확인하였다.

### 테스트 코드

```python
import requests

response = requests.get("http://localhost:8000/health", timeout=30)

print("상태 코드:", response.status_code)
print("응답 내용:", response.json())
```

### 예상 결과

```text
상태 코드: 200
응답 내용: {'status': 'ok', 'model_loaded': True}
```

### 캡처 첨부

![health API 테스트 결과](health_test.png)

### 설명

`/health` API를 통해 서버 상태와 모델 로드 여부를 확인하였다.
`model_loaded` 값이 `True`이면 모델이 정상적으로 로드된 상태이다.

---

## 1.7 `/predict` API 테스트

### 실행 내용

한국어 금융 문장을 입력하여 감정 분석 결과가 반환되는지 확인하였다.

### 테스트 코드

```python
import requests

response = requests.post(
    "http://localhost:8000/predict",
    json={"text": "오늘 주가가 크게 올랐습니다."},
    headers={"X-API-Key": "test-key-001"},
    timeout=60
)

print("상태 코드:", response.status_code)
print("응답 내용:", response.json())
```

### 예상 결과

```text
상태 코드: 200
응답 내용: {'success': True, 'label': 'positive', 'confidence': 0.9...}
```

### 캡처 첨부

![predict API 테스트 결과](predict_test.png)

### 설명

`/predict` API에 문장과 API Key를 함께 보내고, 감정 분석 결과를 확인하였다.
결과에는 성공 여부, 감정 라벨, confidence 점수가 포함된다.

---

## 1.8 API Key 인증 테스트

### 실행 내용

잘못된 API Key를 입력했을 때 요청이 차단되는지 확인하였다.

### 테스트 코드

```python
import requests

response = requests.post(
    "http://localhost:8000/predict",
    json={"text": "오늘 주가가 크게 올랐습니다."},
    headers={"X-API-Key": "wrong-key"},
    timeout=60
)

print("상태 코드:", response.status_code)
print("응답 내용:", response.json())
```

### 예상 결과

```text
상태 코드: 401
응답 내용: {'detail': 'Invalid API Key'}
```

### 캡처 첨부

![API Key 인증 실패 테스트](api_key_test.png)

### 설명

잘못된 API Key를 사용하면 서버가 요청을 거부한다.
이를 통해 인증 기능이 정상적으로 작동함을 확인하였다.

---

## 1.9 Streamlit 실행

### 실행 내용

Streamlit 프론트엔드를 실행하여 사용자 화면을 구성하였다.

### 실행 코드

```python
!streamlit run frontend/app.py --server.port 8501 --server.address 0.0.0.0
```

### 캡처 첨부

![Streamlit 실행 로그](streamlit_log.png)

### 설명

Streamlit 실행 로그에서 `You can now view your Streamlit app in your browser.` 메시지를 확인하였다.
이는 `frontend/app.py`가 정상적으로 실행되었음을 의미한다.

다만 실습 Jupyter 환경의 외부 포트 접근 제한 또는 proxy 경로 문제로 브라우저에서 Streamlit 화면을 직접 여는 것은 제한되었다.
따라서 FastAPI의 `/health`와 `/predict` API 테스트를 통해 백엔드 모델 서빙이 정상 동작함을 확인했고, Streamlit 실행 로그를 통해 프론트엔드 실행 상태를 확인하였다.

---

# 2. 각 섹션 체크포인트 답변

## Q1. 본인의 프로젝트에서 Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?

본 프로젝트에서는 `schemas.py`의 `PredictRequest`에서 Pydantic 검증을 사용하였다.

```python
class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=500
    )
```

이 검증은 다음과 같은 잘못된 입력을 막아준다.

1. `text` 필드가 없는 요청
2. 빈 문자열 입력
3. 500자를 넘는 너무 긴 문장
4. 문자열이 아닌 잘못된 타입의 입력

예를 들어 아래와 같은 요청은 잘못된 입력이다.

```json
{
  "text": ""
}
```

또는 아래와 같이 `text` 필드가 없는 요청도 잘못된 입력이다.

```json
{
  "message": "오늘 주가가 크게 올랐습니다."
}
```

Pydantic은 이런 잘못된 요청을 모델에 전달하기 전에 먼저 검증하여 막아준다.

---

## Q2. `Depends(verify_api_key)`를 제거하면 어떤 위험이 있습니까?

`Depends(verify_api_key)`를 제거하면 API Key 인증이 사라진다.

현재 `/predict` API에는 아래 코드가 적용되어 있다.

```python
user: str = Depends(verify_api_key)
```

이 코드는 API를 호출하는 사용자가 올바른 API Key를 가지고 있는지 확인하는 역할을 한다.

이 부분을 제거하면 다음과 같은 위험이 생긴다.

1. 아무나 API를 호출할 수 있다.
2. 악의적인 사용자가 반복 요청을 보낼 수 있다.
3. 서버 자원이 불필요하게 사용될 수 있다.
4. 모델 추론 비용이 증가할 수 있다.
5. 서비스 안정성이 떨어질 수 있다.

즉, API Key 인증은 서비스 입구의 잠금장치 역할을 한다.

---

## Q3. `run_in_executor`를 사용한 이유는 무엇입니까?

`run_in_executor`를 사용한 이유는 AI 모델 추론 작업이 시간이 오래 걸릴 수 있기 때문이다.

AI 모델은 입력 문장을 분석할 때 계산이 필요하다.
이 작업을 FastAPI의 메인 흐름에서 직접 처리하면 서버가 잠시 멈춘 것처럼 동작할 수 있다.

그래서 아래처럼 `run_in_executor`를 사용하였다.

```python
result = await loop.run_in_executor(
    executor,
    predict,
    model,
    request.text
)
```

쉽게 말하면, 무거운 AI 분석 작업을 서버의 메인 흐름에서 직접 처리하지 않고 별도의 실행 흐름에 맡기는 것이다.

비유하면 FastAPI 서버는 접수 데스크이고, AI 모델 추론은 시간이 걸리는 전문 분석 작업이다.
`run_in_executor`는 접수 데스크가 멈추지 않도록 분석 작업을 옆 작업실에 맡기는 역할을 한다.

---

## Q4. Day 1~8 중 가장 많이 참고한 Day는 어디였습니까? 왜?

가장 많이 참고한 Day는 Day 5와 Day 6이다.

Day 5는 FastAPI 프로젝트 구조를 이해하는 데 도움이 되었다.
특히 `schemas.py`, `model_service.py`, `main.py`처럼 파일 역할을 나누는 구조를 참고하였다.

Day 6은 API Key 인증을 적용하는 데 도움이 되었다.
특히 `Depends(verify_api_key)`를 사용하여 `/predict` API에 인증을 적용하는 부분을 참고하였다.

추가로 Streamlit 화면을 만들 때는 Day 4와 Day 5를 참고하였다.
Streamlit에서 사용자가 입력한 값을 받아 `requests.post()`로 FastAPI 서버에 보내는 구조를 이해하는 데 도움이 되었다.

정리하면 다음과 같다.

* Day 5: FastAPI 프로젝트 구조
* Day 6: API Key 인증
* Day 4/5: Streamlit UI와 API 호출
* Day 3: `run_in_executor`와 에러 처리

---

## Q5. 이 서비스를 실제로 배포하려면 추가로 무엇이 필요합니까?

이 서비스를 실제 사용자에게 배포하려면 다음과 같은 요소가 추가로 필요하다.

1. Docker
2. 클라우드 서버
3. 환경변수 관리
4. 로그 관리
5. 모니터링
6. 보안 강화
7. CI/CD 자동 배포
8. 모델 버전 관리

Docker는 실행 환경을 고정하기 위해 필요하다.
내 컴퓨터나 노트북 환경이 아니라 다른 서버에서도 같은 방식으로 실행되도록 도와준다.

클라우드 서버는 실제 사용자가 접속할 수 있도록 서비스를 외부에 배포하기 위해 필요하다.
예를 들어 AWS, GCP, Azure 같은 클라우드 환경을 사용할 수 있다.

환경변수 관리는 API Key 같은 중요한 값을 코드에 직접 쓰지 않기 위해 필요하다.
실제 배포에서는 `.env` 파일이나 Secret Manager를 사용하는 것이 좋다.

로그와 모니터링은 서비스가 안정적으로 운영되는지 확인하기 위해 필요하다.
사용량, 에러 발생 여부, 응답 시간, 모델 추론 속도 등을 확인할 수 있어야 한다.

CI/CD는 코드를 수정했을 때 자동으로 테스트하고 배포하기 위해 필요하다.
이를 통해 수동으로 서버를 매번 재시작하지 않고 안정적으로 서비스를 운영할 수 있다.

---

# 3. 프로젝트 회고

## 3.1 프로젝트를 통해 배운 점

이번 프로젝트를 통해 AI 모델을 단순히 실행하는 것과 실제 서비스로 만드는 것은 다르다는 것을 배웠다.

모델 하나만 있으면 서비스가 완성되는 것이 아니라, 사용자의 입력을 검증하는 스키마, 요청을 받는 API 서버, 인증 기능, 에러 처리, 사용자 화면이 함께 필요하다는 것을 알게 되었다.

특히 이번 프로젝트에서는 다음과 같은 구조를 직접 경험하였다.

* `schemas.py`: 입력과 출력 형태 정의
* `model_service.py`: 모델 로드와 추론 함수 정의
* `main.py`: FastAPI 서버 구성
* `auth.py`: API Key 인증
* `frontend/app.py`: Streamlit 사용자 화면

이 구조를 통해 AI 서비스를 만들 때 파일별 역할을 분리하는 것이 중요하다는 것을 배웠다.

---

## 3.2 어려웠던 점

가장 어려웠던 부분은 Jupyter Notebook의 `%%writefile` 명령어와 실제 `.py` 파일 코드를 구분하는 것이었다.

`%%writefile`은 노트북 셀의 첫 줄에서만 사용해야 하는 명령어이다.
하지만 이 문장이 실제 `.py` 파일 안에 들어가면 Python은 이를 코드로 해석하려고 하며 `SyntaxError`가 발생한다.

실제로 `main.py`와 `model_service.py` 안에 `%%writefile`이 잘못 들어가서 서버 실행 오류가 발생하였다.
이 문제를 해결하면서 노트북 셀 명령어와 실제 Python 코드의 차이를 이해하게 되었다.

또한 Streamlit 앱은 정상적으로 실행되었지만, 실습 Jupyter 환경의 proxy 또는 외부 포트 접근 제한으로 브라우저에서 직접 접속하는 데 어려움이 있었다.
이 부분을 통해 로컬 개발 환경과 원격 Jupyter 환경의 차이도 알게 되었다.

---

## 3.3 해결 과정

문제를 해결하기 위해 각 파일을 다시 확인하였다.

먼저 `app/main.py`, `app/model_service.py`, `app/schemas.py` 안에 `%%writefile`이 잘못 들어가 있는지 확인하였다.

```bash
grep -R "%%writefile" app frontend
```

그 후 각 파일을 순수한 Python 코드로 다시 저장하였다.

또한 서버 실행 중 에러가 발생했을 때는 `uvicorn`을 직접 실행하여 에러 로그를 확인하였다.
이를 통해 어떤 파일의 몇 번째 줄에서 문제가 발생했는지 확인할 수 있었다.

마지막으로 FastAPI의 `/health`와 `/predict` API를 테스트하여 모델 서빙이 정상 동작하는지 확인하였다.

---

## 3.4 다음에 다시 만든다면 개선하고 싶은 점

다음에 다시 만든다면 처음부터 더 작은 모델로 API 구조를 먼저 완성한 뒤, 이후에 원하는 모델로 교체하고 싶다.

또한 각 파일을 작성한 직후 바로 import 테스트와 API 테스트를 수행할 것이다.
예를 들어 `schemas.py`를 작성한 뒤에는 바로 아래와 같은 테스트를 실행할 것이다.

```python
from app.schemas import PredictRequest, PredictResponse
```

`model_service.py`를 작성한 뒤에는 아래와 같이 테스트할 것이다.

```python
from app.model_service import load_model, predict
```

이렇게 하면 문제가 생겼을 때 어느 파일에서 문제가 발생했는지 더 빠르게 찾을 수 있다.

실제 배포 단계에서는 Docker를 사용해 실행 환경을 고정하고, 클라우드 서버에 배포하며, CI/CD 파이프라인을 통해 자동 테스트와 자동 배포가 가능하도록 발전시키고 싶다.

---

## 3.5 최종 소감

이번 프로젝트는 Day 1~7에서 배운 내용을 종합하여 나만의 AI 서비스를 만들어 보는 과정이었다.

처음에는 파일도 많고 서버와 프론트엔드가 나뉘어 있어 어렵게 느껴졌다.
하지만 하나씩 나누어 보니 각 파일의 역할을 이해할 수 있었다.

이번 프로젝트를 통해 AI 서비스 개발은 모델만 다루는 것이 아니라, 서버, 인증, 입력 검증, 에러 처리, 사용자 화면까지 함께 고려해야 한다는 것을 배웠다.

아직 완벽하게 혼자 모든 코드를 작성하기는 어렵지만, 이번 과정을 통해 AI 모델을 서비스 형태로 연결하는 전체 흐름을 이해할 수 있었다.


![health 테스트 결과](health_test.png)

![predict 테스트 결과](predict_test.png)

![FastAPI 테스트 결과](FastAPI_test.png)